# AML Alert Triage & Risk Prioritization

This notebook analyses a synthetic AML alert dataset and develops a transparent rule-based priority score to support risk-based triage and operational decision-making.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


## 1. Load and inspect the dataset


In [ ]:
df = pd.read_csv("aml_alerts_synthetic.csv")


In [ ]:
display(df.head())
print('Shape:', df.shape)
df.info()
print('\nMissing values:')
display(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())


## 2. Data preparation

Duplicate records are removed, missing numerical values are filled with the median, and missing categorical risk indicators are retained as `Unknown` so that they can still be handled explicitly in the prioritisation logic.


In [26]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Fill missing numerical values with the median
df["transaction_amount"] = df["transaction_amount"].fillna(
    df["transaction_amount"].median()
)

# Fill missing categorical values
df["country_risk"] = df["country_risk"].fillna("Unknown")
df["adverse_media_flag"] = df["adverse_media_flag"].fillna("Unknown")

# Validate cleaned dataset
print(df.isnull().sum())
print(df.shape)

alert_id                0
customer_id             0
customer_type           0
country                 0
country_risk            0
customer_risk_rating    0
alert_type              0
alert_date              0
transaction_amount      0
previous_alerts_90d     0
pep_flag                0
adverse_media_flag      0
case_status             0
dtype: int64
(795, 13)


## 3. Risk indicator review


In [28]:
df['customer_risk_rating'].value_counts()

,count
customer_risk_rating,
Medium,314
Low,287
High,194


In [32]:
df['country_risk'].value_counts(dropna=False)

,count
country_risk,
Low,321
Medium,234
High,225
Unknown,15


In [36]:
df['pep_flag'].value_counts(dropna=False)

,count
pep_flag,
No,742
Yes,53


In [38]:
df['adverse_media_flag'].value_counts(dropna=False)

,count
adverse_media_flag,
No,691
Yes,94
Unknown,10


In [40]:
df['previous_alerts_90d'].describe()

,previous_alerts_90d
count,795.000000
mean,1.315723
std,1.623099
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,9.000000


## 4. Rule-based priority scoring

The priority score combines five risk indicators:

- Customer risk rating
- Country risk
- PEP status
- Adverse media
- Previous alerts in the last 90 days

The model is intentionally transparent and rule-based rather than predictive. Higher-risk characteristics receive more points, and the final score is used to prioritise alerts for review.


In [29]:
df['priority_score']=0

In [30]:
df['priority_score'] = df['priority_score'] + df['customer_risk_rating'].map({
    'Low': 2,
    'Medium': 5,
    'High': 8
})

In [33]:
df['priority_score'] = df['priority_score'] + df['country_risk'].map({
    'Low': 1,
    'Medium': 3,
    'High': 5,
    'Unknown': 4
})

In [37]:
df['priority_score'] = df['priority_score'] + df['pep_flag'].map({
    'No': 0,
    'Yes': 4,

})

In [39]:
df['priority_score'] = df['priority_score'] + df['adverse_media_flag'].map({
    'No': 0,
    'Yes': 5,
    'Unknown': 2,

})

In [41]:
def score_previous_alerts(x):
    if x == 0:
        return 0
    elif x <= 2:
        return 1
    else:
        return 2

df['previous_alerts_score'] = df['previous_alerts_90d'].apply(score_previous_alerts)

In [42]:
df['priority_score'] = df['priority_score'] + df['previous_alerts_score']

In [44]:
df.groupby('previous_alerts_90d')['previous_alerts_score'].unique()

,previous_alerts_score
previous_alerts_90d,
0,[0]
1,[1]
2,[1]
3,[2]
4,[2]
5,[2]
6,[2]
7,[2]
8,[2]


## 5. Priority score validation


In [45]:
df['priority_score'].describe()

,priority_score
count,795.000000
mean,9.080503
std,4.096575
min,3.000000
25%,6.000000
50%,9.000000
75%,12.000000
max,21.000000


In [ ]:
print("Missing priority scores:", df['priority_score'].isna().sum())


## 6. Priority segmentation

Alerts are grouped into Low, Medium and High priority bands based on the observed score distribution.


In [67]:
df['priority_level'] = pd.cut(
    df['priority_score'],
    bins=[3, 6, 12, 21],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)

In [68]:
df['priority_level'].value_counts()

,count
priority_level,
Medium,359
Low,253
High,183


In [ ]:
order = ['Low', 'Medium', 'High']
counts = df['priority_level'].value_counts().reindex(order)

ax = counts.plot.bar()
ax.set_title('AML Alerts by Priority Level')
ax.set_xlabel('Priority Level')
ax.set_ylabel('Number of Alerts')
plt.tight_layout()
plt.savefig('priority_level_distribution.png', bbox_inches='tight')
plt.show()


## 7. Highest-priority alerts

The highest-scoring alerts are reviewed as an operational triage output, showing which cases would rise to the top of a risk-based review queue.


In [75]:
top_10_alerts = df.sort_values(
    by='priority_score',
    ascending=False
).head(10)

top_10_alerts[[
    'alert_id',
    'customer_id',
    'alert_type',
    'transaction_amount',
    'customer_risk_rating',
    'country_risk',
    'pep_flag',
    'adverse_media_flag',
    'previous_alerts_90d',
    'priority_score',
    'priority_level'
]]

,alert_id,customer_id,alert_type,transaction_amount,customer_risk_rating,country_risk,pep_flag,adverse_media_flag,previous_alerts_90d,priority_score,priority_level
266,ALT-0267,CUST-0213,Structuring Pattern,9979.56,Medium,High,Yes,Yes,3,21,High
135,ALT-0136,CUST-0080,Large Transaction,146096.94,High,High,No,Yes,3,20,High
190,ALT-0191,CUST-0352,Large Transaction,821.62,High,High,No,Yes,6,20,High
96,ALT-0097,CUST-0235,Structuring Pattern,11301.61,High,High,No,Yes,2,19,High
443,ALT-0444,CUST-0182,Structuring Pattern,25300.46,High,High,Yes,No,4,19,High
239,ALT-0240,CUST-0072,Large Transaction,12856.36,High,High,No,Yes,2,19,High
201,ALT-0202,CUST-0046,High-Risk Counterparty,30038.11,High,High,No,Yes,2,19,High
319,ALT-0320,CUST-0374,Large Transaction,3109.94,High,High,Yes,No,1,18,High
530,ALT-0531,CUST-0341,Structuring Pattern,18866.00,High,High,Yes,No,1,18,High
542,ALT-0543,CUST-0220,Dormant Account Activity,5317.67,High,High,No,Yes,0,18,High


## 8. Business findings


In [76]:
high_priority = df[df['priority_level'] == 'High']

print("High-priority alerts:", len(high_priority))
print("High-priority %:", round(len(high_priority) / len(df) * 100, 1), "%")

print(
    "PEP Yes %:",
    round((high_priority['pep_flag'] == 'Yes').mean() * 100, 1),
    "%"
)

print(
    "Adverse Media Yes %:",
    round((high_priority['adverse_media_flag'] == 'Yes').mean() * 100, 1),
    "%"
)

print(
    "High Country Risk %:",
    round((high_priority['country_risk'] == 'High').mean() * 100, 1),
    "%"
)

print(
    "Median previous alerts:",
    high_priority['previous_alerts_90d'].median()
)

print(
    "Median transaction amount:",
    round(high_priority['transaction_amount'].median(), 2)
)

High-priority alerts: 183
High-priority %: 23.0 %
PEP Yes %: 12.6 %
Adverse Media Yes %: 31.7 %
High Country Risk %: 78.7 %
Median previous alerts: 1.0
Median transaction amount: 8967.5


In [77]:
print(
    "Overall High Country Risk %:",
    round((df['country_risk'] == 'High').mean() * 100, 1),
    "%"
)

print(
    "Overall PEP Yes %:",
    round((df['pep_flag'] == 'Yes').mean() * 100, 1),
    "%"
)

print(
    "Overall Adverse Media Yes %:",
    round((df['adverse_media_flag'] == 'Yes').mean() * 100, 1),
    "%"
)

print(
    "Overall median previous alerts:",
    df['previous_alerts_90d'].median()
)

print(
    "Overall median transaction amount:",
    round(df['transaction_amount'].median(), 2)
)

Overall High Country Risk %: 28.3 %
Overall PEP Yes %: 6.7 %
Overall Adverse Media Yes %: 11.8 %
Overall median previous alerts: 1.0
Overall median transaction amount: 9101.57


### Interpretation

- **183 alerts (23.0%)** are classified as High priority, creating a focused population for risk-based review.
- **78.7% of High-priority alerts involve High-risk countries**, compared with **28.3% overall**.
- **31.7% of High-priority alerts have adverse media**, compared with **11.8% overall**.
- **12.6% of High-priority alerts involve PEPs**, compared with **6.7% overall**.
- Median previous alerts are **1** in both the High-priority group and the full dataset, so repeated alerts alone are not the main driver of prioritisation.
- Median transaction amount is similar for High-priority alerts (**8,967.50**) and the overall population (**9,101.57**), suggesting that prioritisation is driven more by the combination of risk characteristics than by transaction size alone.

Because the same risk indicators are used to construct the rule-based score, these comparisons describe the composition of the prioritised population rather than proving causal relationships.
